Fazendo a minha primeira chamada da LLM 

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv 
import os

load_dotenv()

True

Criando a primeira chamada de LLM

In [56]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=os.getenv("GOOGLE_API_KEY"))  

In [57]:
llm.invoke("Quem foi Maria Antonieta?").content

'**Maria Antonieta (nascida Maria Antonia Josepha Johanna von Habsburg-Lothringen)** foi uma arquiduquesa austríaca que se tornou Rainha da França e Navarra como esposa do Rei Luís XVI. Ela é uma das figuras mais emblemáticas da história francesa, conhecida por seu estilo de vida luxuoso, sua impopularidade e seu trágico fim durante a Revolução Francesa.\n\nAqui estão os pontos chave sobre sua vida:\n\n1.  **Origem e Casamento Político:**\n    *   Nascida em 2 de novembro de 1755 em Viena, Áustria, era a décima quinta e penúltima filha da imperatriz Maria Teresa da Áustria e do imperador Francisco I do Sacro Império Romano-Germânico.\n    *   Seu casamento, aos 14 anos, em 1770, com o Delfim Luís Augusto (futuro Luís XVI) foi uma aliança política destinada a fortalecer os laços entre as casas reais da Áustria e da França, que haviam sido inimigas por séculos.\n\n2.  **Vida como Delfina e Rainha:**\n    *   Como Delfina e, a partir de 1774, como Rainha da França, Maria Antonieta rapidam

Construindo a primeira estrutura de RAG

In [58]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
df =  pd.read_excel(DOCS_DIR / "Tickets.xlsx")
from langchain_core.documents import Document
BASE_DIR = Path().resolve()
DOCS_DIR = BASE_DIR / "docs"

Escolhendo a estrutura de embedding

In [ ]:
import os
os.environ["TRANSFORMERS_NO_LAZY_IMPORT"] = "1"
import transformers.models.bert
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:

def load_pdf_vectorstore(filepath: str, save_path: str):
    loader = PyPDFLoader(DOCS_DIR / filepath)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=500)
    documents = text_splitter.split_documents(documents)
    vectorstore = FAISS.from_documents(documents, embedding)    
    vectorstore.save_local(f'vectostores/{save_path}')
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7}) 
    return retriever

In [ ]:
retriever_perguntas_frequentes = load_pdf_vectorstore("Perguntas Frequentes.pdf", "vectorstore_perguntas_frequentes")
retriever_manual_tecnico = load_pdf_vectorstore("Manual Tecnico de Produtos.pdf", "vectorstore_manual_tecnico_produtos")
retriever_politicas_procedimentos = load_pdf_vectorstore("Politicas e Procedimentos.pdf", "vectorstore_politica_procedimentos")   

In [ ]:
def load_excel_vectorstore(filepath: str, save_path: str):
    df =  pd.read_excel(DOCS_DIR / filepath)
    documents = []
    for idx, row in df.iterrows():
        text  = ' '.join([str(cell) for cell in row if pd.notna(cell)])
        documents.append(Document(page_content=text, metadata={'row': idx}))
      
    vectorstore_tickets = FAISS.from_documents(documents, embedding)    
    vectorstore_tickets.save_local('vectostores/vectorstore_tickets')
    retriever_tickets = vectorstore_tickets.as_retriever(search_type="similarity", search_kwargs={"k": 7})
    
    return retriever_tickets

In [ ]:
retriever_tickets = load_excel_vectorstore("Tickets.xlsx", "vectorstore_tickets")

Criando os agentes assistentes

In [1]:
#importando bibliotecas
from typing import TypedDict, Optional , List #tipando os dados
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage #definindo o formato da mensagem

In [2]:
class State(TypedDict, total=False): #herdando de TypedDict e total=False para permitir chaves opcionais
  query: str
  route : Optional[str] #rota é opcional, pode ser str ou None
  anwser: Optional[str] #resposta é opcional, pode ser str ou None
  chat_history: Optional[List[BaseMessage]] #histórico de mensagens é opcional, pode ser uma lista de mensagens ou None   
  

In [ ]:
def agent_with_retriever(state: State, papel: str, prompt_instructions: str, retriever = None):
    query = state["query"] #recuperar pergunta feita pelo usuário
    chat_history = state.get("chat_history", []) #recuperar histórico de mensagens, se não existir, usar lista vazia
    context = "" #inicializar contexto vazio
   
    if retriever:
        recuperados = retriver.get_relevant_documents(query) #recuperar documentos relevantes usando o retriever
        if recuperados: #se houver documentos recuperados, construir o contexto concatenando o conteúdo das páginas
            context = "\n".join([doc.page_content for doc in recuperados]) #construir o contexto concatenando o conteúdo das páginas recuperadas, separando por nova linha

  #criando estrutura de mensagens. A primeira mensagem é do sistema, definindo o papel do agente e as instruções. Em seguida, adicionamos o histórico de mensagens da conversa, se houver, e por fim, a pergunta atual do usuário junto com o contexto recuperado.
    mensagens = [
        SystemMessage(content=
                      f"Você é um {papel}."
                      f"Suas instruoes:\n{prompt_instructions}\n\n"
                      f" - Use sempre o contexto recuperado para responder a última pergunta do usuário.\n"
                      f" - Use o histórico da conversa para entender o contexto geral e perguntas de acompanhamento. \n"
                      f" - Se nao houver informaçoes relevantes no contexto, diga que nao encontrou dados suficientes para responder a pergunta. \n"
                      f" - Evite inventar informações."
                 ),
                *chat_history, #desempacotar o histórico de mensagens e adicioná-lo à lista de mensagens
          HumanMessage(content=(
               f"Pergunta do usuário: \n{query}\n\n"
               f"Contexto disponivel para esta pergunta: \n{context if context else 'Nenhum contexto disponível.'}"
           )) #adicionar a pergunta atual do usuário como uma mensagem do tipo
    ]
    resposta = llm.invoke(mensagens)#gerar resposta usando o modelo de linguagem, passando a lista de mensagens como entrada
    state["anwser"] = resposta.content #atualizar o estado com a resposta gerada
    return state

Criando o agente de detalhe técnico

In [ ]:
def agent_detalhe_tecnico(state: State):
    prompt_instructions = (
        "Seja um **especialista em suporte técnico e produto**. " 
        "Você deve responder a perguntas sobre **especificações técnicas**."
        "**instrucoes de instalação**, **manutençao preventiva** e **soluçao de problemas**."
        "Sua resposta deve ser precisa, técnica e objetiva, baseada estritamente no manualk técnico."
        "Para problemas, ofereça uma soluçao clara e passo a passo."
    )
    return agent_with_retriever(
        state, "especialista em detalhes técnicos de produtos", prompt_instructions, retriever_manual_tecnico    )

Criando o agente de perguntas frequentes

In [ ]:
def agent_detalhe_tecnico(state: State):
    prompt_instructions = (
        "Seja um **especialista em suporte técnico e produto**. " 
        "Você deve responder a perguntas sobre **especificações técnicas**."
        "**instrucoes de instalação**, **manutençao preventiva** e **soluçao de problemas**."
        "Sua resposta deve ser precisa, técnica e objetiva, baseada estritamente no manualk técnico."
        "Para problemas, ofereça uma soluçao clara e passo a passo."
    )
    return agent_with_retriever(
        state, "especialista em detalhes técnicos de produtos", prompt_instructions, retriever_manual_tecnico    )

criando o agente de politicas e procedimentos

In [ ]:
def agent_detalhe_tecnico(state: State):
    prompt_instructions = (
        "Seja um **especialista em suporte técnico e produto**. " 
        "Você deve responder a perguntas sobre **especificações técnicas**."
        "**instrucoes de instalação**, **manutençao preventiva** e **soluçao de problemas**."
        "Sua resposta deve ser precisa, técnica e objetiva, baseada estritamente no manualk técnico."
        "Para problemas, ofereça uma soluçao clara e passo a passo."
    )
    return agent_with_retriever(
        state, "especialista em detalhes técnicos de produtos", prompt_instructions, retriever_manual_tecnico    )

Criando o agente de tickets

In [ ]:
def agent_detalhe_tecnico(state: State):
    prompt_instructions = (
        "Seja um **especialista em suporte técnico e produto**. " 
        "Você deve responder a perguntas sobre **especificações técnicas**."
        "**instrucoes de instalação**, **manutençao preventiva** e **soluçao de problemas**."
        "Sua resposta deve ser precisa, técnica e objetiva, baseada estritamente no manualk técnico."
        "Para problemas, ofereça uma soluçao clara e passo a passo."
    )
    return agent_with_retriever(
        state, "especialista em detalhes técnicos de produtos", prompt_instructions, retriever_manual_tecnico    )